In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, TransformerMixin

from tqdm import tqdm

from sklearn.pipeline import Pipeline

from sklearn.ensemble import GradientBoostingRegressor




from skorch import NeuralNetRegressor

In [2]:
df_final = pd.read_excel("data/processed/df_final.xlsx")
df_final = df_final.sort_values("Date").reset_index(drop=True) 

In [3]:
covariates = ["dolvol_lag2", "maxret", "retvol", "mom36m", "mom12m", "mom6m", "mom1m", "chmom", "turn", "indmom", "baspread", "illiq", "stdturn", "beta", "beta_squared",
"idiovol", "mvel1", "agr", "cashpr", "chinv", "chsh", "depr", "dy", "ep", "invest", "rd_mve", "sp", "nincr"]

I. Fonctions : normalisation et gestion des NaN

In [4]:
def generate_time_splits(df, date_col='Date',
                                   val_months=12,
                                   test_months=12,
                                   step_months=12,
                                   min_train_months=306):
    """
    Génère des splits temporels selon la logique décrite :
    - Train cumulatif (augmente d'un an à chaque refit)
    - Validation = fenêtre fixe glissante de 1 an 
    - Test = fenêtre fixe après la validation
    - Avance de step_months à chaque itération : 12 mois

    Paramètres :
    - df : DataFrame trié par date
    - date_col : nom de la colonne des dates
    - val_months : taille de la validation
    - test_months : taille du test 
    - step_months : pas de glissement
    - min_train_months : nombre minimum de mois de train initial

    Retour :
    - splits : liste de tuples (train_idx, val_idx, test_idx)
    """

    #On coupe chronologiquement donc on trie par date 
    df = df.sort_values(date_col).reset_index(drop=True)
    dates = sorted(df[date_col].unique())
    total_months = len(dates)

    splits = []

    #On démarre après avoir au moins min_train_months pour le train
    start = min_train_months
    while True:
        train_end = start  # train va de 0 jusqu'à train_end
        val_start = train_end
        val_end = val_start + val_months
        test_start = val_end
        test_end = test_start + test_months

        #Stop si on n'a plus assez pour test
        if test_end > total_months:
            break

        train_dates = dates[:train_end]  
        val_dates = dates[val_start:val_end]
        test_dates = dates[test_start:test_end]

        train_idx = df[df[date_col].isin(train_dates)].index.tolist()
        val_idx = df[df[date_col].isin(val_dates)].index.tolist()
        test_idx = df[df[date_col].isin(test_dates)].index.tolist()

        splits.append((train_idx, val_idx, test_idx))

        #Avancer d'un step (ex : 12 mois) pour le prochain refit
        start += step_months

    return splits

In [ ]:
def preprocess_split(X_train, X_val, X_test, covariates):
    """
    Impute les NaN par moyenne par Ticker (fit sur train),
    puis normalise chaque covariable entre -1 et 1 par date (rang cross-sectionnel).

    Paramètres
    ----------
    X_train, X_val, X_test : DataFrames bruts (avec 'Ticker' et 'Date')
    covariates : liste des colonnes numériques à traiter

    Retour
    ------
    X_train_scaled, X_val_scaled, X_test_scaled : DataFrames transformés (covariates seulement)
    """
    #Gestion des NaN : moyenne par Ticker calculée sur le train
    means_by_ticker = x_train.groupby("Ticker")[covariates].mean(numeric_only=True)

    def fill_na_with_means(df):
        df = df.copy()
        for col in covariates:
            #On remplace NaN par la moyenne du ticker
            df[col] = df.apply(
                lambda row: means_by_ticker[col][row["Ticker"]] 
                            if pd.isna(row[col]) and row["Ticker"] in means_by_ticker.index 
                            else row[col],
                axis=1
            )
        return df

    x_train_imp = fill_na_with_means(x_train)
    x_val_imp   = fill_na_with_means(x_val)
    x_test_imp  = fill_na_with_means(x_test)

    #Normalisation cross-sectionnelle : par date
    def normalize_by_date(df):
        df = df.copy()
        out = pd.DataFrame(index=df.index, columns=covariates)
        for date_key, group_idx in df.groupby("Date").groups.items():
            sub = df.loc[group_idx, covariates]
            for cov in covariates:
                temp = sub[cov].dropna().sort_values()
                n = len(temp)
                if n == 1:
                    scores = pd.Series([0.0], index=temp.index)
                else:
                    scores = pd.Series(
                        2 * np.arange(n) / (n - 1) - 1, index=temp.index
                    )
                out.loc[temp.index, cov] = scores
        return out.astype(float)

    x_train_scaled = normalize_by_date(x_train_imp)
    x_val_scaled   = normalize_by_date(x_val_imp)
    x_test_scaled  = normalize_by_date(x_test_imp)

    #garde l'information de la date et du ticker
    x_train_scaled = pd.concat([x_train_imp[['Ticker','Date']].reset_index(drop=True), x_train_scaled.reset_index(drop=True)], axis=1)
    x_val_scaled   = pd.concat([x_val_imp[['Ticker','Date']].reset_index(drop=True), x_val_scaled.reset_index(drop=True)], axis=1)
    x_test_scaled  = pd.concat([x_test_imp[['Ticker','Date']].reset_index(drop=True), x_test_scaled.reset_index(drop=True)], axis=1)

    return x_train_scaled, x_val_scaled, x_test_scaled

In [ ]:
#Mesures : 
#R²
def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum(y_true**2) 
    return 1 - ss_res/ss_tot if ss_tot != 0 else np.nan


In [7]:
#Permet de récupérer x et y 
def get_x_y(df, idx, target="excess_return"):
    subset = df.loc[idx].copy()
    x = subset.drop(columns=[target]) #garde toutes les colonnes mais enlève excess return
    y = subset[target]
    return x, y

In [ ]:
#On découpe les splits puis on applique la gestion des NaN et la normalisation définie plus haut
splits = generate_time_splits(df_final)

preprocessed_splits = []

for train_idx, val_idx, test_idx in tqdm(splits):
    x_train, y_train = get_x_y(df_final, train_idx)
    x_val, y_val = get_x_y(df_final, val_idx)
    x_test, y_test = get_x_y(df_final, test_idx)

    #Imputation + Normalisation
    x_train, x_val, x_test = preprocess_split(x_train, x_val, x_test, covariates) #on enlève ticker et date

    preprocessed_splits.append((x_train, y_train, x_val, y_val, x_test, y_test))

100%|██████████| 3/3 [01:33<00:00, 31.14s/it]


BENCHMARK

In [ ]:
#historical average benchmark
ha_pred = []
ha_true = []
for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    y_trainval = pd.concat([y_train, y_val])
    mean_hist = y_trainval.mean() 
    y_pred_ha = np.full_like(y_test, fill_value=mean_hist, dtype=float)

    ha_pred.extend(y_pred_ha)
    ha_true.extend(y_test)
    
ha_true = np.array(ha_true)
ha_pred = np.array(ha_pred)
r2_ha_oos = r2(ha_true, ha_pred)

MSFE_ha = mean_squared_error(ha_pred, ha_true)
print(len(ha_pred), len(ha_true))

r2_results = {
  "r2_ha_oos" : r2_ha_oos
}

print(r2_ha_oos)

100%|██████████| 3/3 [00:00<00:00, 286.24it/s]

2135 2135
0.008474980675856725


In [33]:
"""
OLS : Ordinary Least Squares : Nous détaillons ici cet algorithme, la logique étant identique pour les autres modèles.
Listes utilisées : 
- r2_in_sample_list et r2_test_list : stockent, pour chaque split, les R² in‑sample et out‑of‑sample. Elles servent à analyser
  la performance split par split et à ajuster le tuning des modèles (utile pour les modèles à hyperparamètres).
- y_true : valeurs réelles de l’equity premium sur l’ensemble.
- y_pred_ols : prédictions correspondantes du modèle OLS. 
- y_trainval : données d’entraînement (train + validation) utilisées pour l’ajustement du modèle.
- dates_ols et tickers_ols : récupérées à chaque split pour pouvoir fusionner correctement les prédictions de tous les modèles
  et s’assurer que les lignes (dates/tickers) correspondent, évitant tout mélange potentiel des prédictions.

Df et résultats en sortie : 
- df_results_ols : df contenant les prédictions du modèles ols ainsi que la date et le ticker correspondant. 
- r2_results : dictionnaire contenant le r2 ols in sample et oos
"""

r2_in_sample_list_ols = []
r2_test_list_ols = []

y_true = []
y_pred_ols = []

y_trainval_true_ols = []
y_trainval_pred_ols = []

dates_ols = []
tickers_ols = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    ols = LinearRegression()
    ols.fit(x_trainval, y_trainval)

    #R² in-sample
    y_trainval_pred = ols.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_sample_list_ols.append(r2_in)

    #R² oos
    y_test_pred = ols.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_test_list_ols.append(r2_out)


    #On stocke tout dans un tableau
    tickers_ols.append(x_test["Ticker"])
    dates_ols.append(x_test["Date"])
    y_true.append(y_test)
    y_pred_ols.append(y_test_pred)
    y_trainval_true_ols.append(y_trainval)
    y_trainval_pred_ols.append(y_trainval_pred) 

    print(f"R² in-sample : {r2_in:.6f} | R² test : {r2_out:.6f}")

#On concatène les résultats de tous les splits en un df
dates_ols = np.concatenate(dates_ols)
tickers_ols = np.concatenate(tickers_ols)
y_true = np.concatenate(y_true)
y_pred_ols = np.concatenate(y_pred_ols)

y_trainval_true_ols = np.concatenate(y_trainval_true_ols)
y_trainval_pred_ols = np.concatenate(y_trainval_pred_ols)

df_results_ols = pd.DataFrame({
    "Date": dates_ols,
    "Ticker": tickers_ols, 
    "y_true": y_true,
    "y_pred_ols": y_pred_ols,
})

r2_insample_ols = r2(y_trainval_true_ols, y_trainval_pred_ols)
r2_oos_ols = r2(y_true, y_pred_ols)

r2_results = {
    "r2_in_sample_ols": r2_insample_ols,
    "r2_oos_global_ols": r2_oos_ols
}

  0%|          | 0/3 [00:00<?, ?it/s]

 67%|██████▋   | 2/3 [00:00<00:00,  7.12it/s]

R² in-sample : 0.037968 | R² test : 0.013748
R² in-sample : 0.037624 | R² test : 0.031366


100%|██████████| 3/3 [00:00<00:00,  7.05it/s]

R² in-sample : 0.037495 | R² test : 0.023614


ALGORITHMES

In [ ]:
"""
PLS : Partial Least Squares : Contrairement à l’OLS, les modèles qui suivent nécessitent de rechercher des hyperparamètres.

Hyperparamètres : 
- k : nombre de composantes latentes sélectionnées par le PLS. On teste différents k et on retient celui qui minimise 
MSE (Mean Squared Error)

Listes utilisées (autre que celles déjà présentées): 
- best_components_list : stocke, pour chaque split, la valeur optimale de k retenue.
- mse_val_grids : pour chaque split, on teste différents k dans le modèle PLS et on calcule la MSE pour chacun. Cette liste 
contient donc, pour chaque split, toutes les valeurs de MSE obtenues, ce qui permet d’identifier le k qui minimise la MSE.

Tous les hyperparamètres sont calculés comme ceci. 
"""

r2_in_sample_list_pls = []
r2_test_list_pls = []
best_components_list = [] #dictionnaire pour récupérer les meilleurs paramètres à chaque spit
mse_val_grids = []

dates_pls = []
tickers_pls = []
y_pred_pls = []

y_trainval_true_pls = []
y_trainval_pred_pls = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    candidate_ks = np.arange(1, 28) #on prend 28 car on a 28 variables
    mse_val_grid = []
    best_mse = float('inf')
    best_k = None

    #pour chaque k on fait un PLS
    for k in candidate_ks: 
        pls = PLSRegression(n_components=k, scale=False)
        pls.fit(x_train[covariates], y_train)
        y_val_pred = pls.predict(x_val[covariates]).ravel()
        mse_val = mean_squared_error(y_val, y_val_pred)
        mse_val_grid.append(mse_val)

        if mse_val < best_mse: #on compare MSE du modèle en cours avec la meilleure MSE, permet de gader le k qui minimise MSE
            best_mse = mse_val
            best_k = k

    mse_val_grids.append(mse_val_grid)
    best_components_list.append(best_k)
    print(f"Split {split_idx} : meilleur nombre de composantes k = {best_k}") 

    #Réentraîner sur échantillon in-sample avec le meilleur k
    pls_final = PLSRegression(n_components=best_k, scale=False)
    pls_final.fit(x_trainval, y_trainval)

    #R² in sample
    y_trainval_pred = pls_final.predict(x_trainval).ravel()
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_sample_list_pls.append(r2_in)

    #R² oos
    y_test_pred = pls_final.predict(x_test[covariates]).ravel()
    r2_out = r2(y_test, y_test_pred)
    r2_test_list_pls.append(r2_out)

    print(f"R² in-sample : {r2_in:.6f} | R² test : {r2_out:.6f}")


    dates_pls.append(x_test["Date"])
    tickers_pls.append(x_test["Ticker"])
    y_pred_pls.append(y_test_pred)
    y_trainval_true_pls.append(y_trainval)
    y_trainval_pred_pls.append(y_trainval_pred)

dates_pls = np.concatenate(dates_pls)
y_pred_pls = np.concatenate(y_pred_pls)
tickers_pls= np.concatenate(tickers_pls)

y_trainval_true_pls = np.concatenate(y_trainval_true_pls)
y_trainval_pred_pls = np.concatenate(y_trainval_pred_pls) 

df_results_pls = pd.DataFrame({
    "Date": dates_pls,
    "Ticker" : tickers_pls,
    "y_pred_pls": y_pred_pls
})

r2_insample_pls = r2(y_trainval_true_pls, y_trainval_pred_pls)
r2_oos_pls = r2(y_true, y_pred_pls)

print(r2_insample_pls)
print(f"Moyenne des R² in-sample : {np.mean(r2_in_sample_list_pls):.6f}")

r2_results = {
    "r2_in_sample_pls": r2_insample_pls,
    "r2_oos_global_pls": r2_oos_pls
}

 33%|███▎      | 1/3 [00:07<00:15,  7.97s/it]

Split 1 : meilleur nombre de composantes k = 1
R² in-sample : 0.022604 | R² test : 0.029389


 67%|██████▋   | 2/3 [00:14<00:07,  7.34s/it]

Split 2 : meilleur nombre de composantes k = 4
R² in-sample : 0.031395 | R² test : 0.020604


100%|██████████| 3/3 [00:19<00:00,  6.55s/it]

Split 3 : meilleur nombre de composantes k = 12
R² in-sample : 0.037491 | R² test : 0.023637
0.03060820291622446
Moyenne des R² in-sample : 0.030497


In [ ]:
"""
PLS : Principal Component Regression : Suit la même logique que PCR et utilisent le même hyperparamètre k.
"""

r2_in_sample_list_pcr = []
r2_test_list_pcr = []
best_components_pcr = []
mse_val_grids_pcr = []

dates_pcr = []
tickers_pcr = []
y_pred_pcr = []

y_trainval_true_pcr = []
y_trainval_pred_pcr = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    candidate_ks = np.arange(1, 28)
    mse_val_grid = []
    best_mse = float('inf')
    best_k = None

    for k in candidate_ks:
        pcr_pipe = Pipeline([
            ('pca', PCA(n_components=k)),
            ('reg', LinearRegression())
        ])
        pcr_pipe.fit(x_train[covariates], y_train)
        y_val_pred = pcr_pipe.predict(x_val[covariates]).ravel()
        mse_val = mean_squared_error(y_val, y_val_pred)
        mse_val_grid.append(mse_val)

        if mse_val < best_mse:
            best_mse = mse_val
            best_k = k

    mse_val_grids_pcr.append(mse_val_grid)
    best_components_pcr.append(best_k)
    print(f"Split {split_idx} : meilleur nombre de composantes k = {best_k}")

    pcr_final = Pipeline([
        ('pca', PCA(n_components=best_k)),
        ('reg', LinearRegression())
    ])
    pcr_final.fit(x_trainval, y_trainval)

    y_trainval_pred = pcr_final.predict(x_trainval).ravel()
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_sample_list_pcr.append(r2_in)

    y_test_pred = pcr_final.predict(x_test[covariates]).ravel()
    r2_out = r2(y_test, y_test_pred)
    r2_test_list_pcr.append(r2_out)

    print(f"R² in-sample : {r2_in:.6f} | R² test : {r2_out:.6f}")

    tickers_pcr.append(x_test["Ticker"])
    dates_pcr.append(x_test["Date"])
    y_pred_pcr.append(y_test_pred)
    y_trainval_true_pcr.append(y_trainval)
    y_trainval_pred_pcr.append(y_trainval_pred)

tickers_pcr = np.concatenate(tickers_pcr)
dates_pcr = np.concatenate(dates_pcr)
y_pred_pcr = np.concatenate(y_pred_pcr)
y_trainval_true_pcr = np.concatenate(y_trainval_true_pcr)
y_trainval_pred_pcr = np.concatenate(y_trainval_pred_pcr)

df_results_pcr = pd.DataFrame({
    "Date": dates_pcr,
    "Ticker": tickers_pcr, 
    "y_pred_pcr": y_pred_pcr
})

r2_insample_pcr = r2(y_trainval_true_pcr, y_trainval_pred_pcr)
r2_oos_pcr = r2(y_true, y_pred_pcr)

r2_results = {
    "r2_in_sample_pcr": r2_insample_pcr,
    "r2_oos_global_pcr": r2_oos_pcr
}

 33%|███▎      | 1/3 [00:00<00:01,  1.07it/s]

Split 1 : meilleur nombre de composantes k = 4
R² in-sample : 0.019830 | R² test : 0.026236


 67%|██████▋   | 2/3 [00:02<00:01,  1.04s/it]

Split 2 : meilleur nombre de composantes k = 24
R² in-sample : 0.028834 | R² test : 0.018903


100%|██████████| 3/3 [00:03<00:00,  1.01s/it]

Split 3 : meilleur nombre de composantes k = 27
R² in-sample : 0.037495 | R² test : 0.023614


In [ ]:
"""
Enet : Elastic Net 

Hyperparamètres : 
- lamba : coefficient de pénalisation
- p : est fixé à 0.5
"""

r2_in_sample_list_en = []
r2_test_list_en = []
best_lambdas = [] 

dates_en = []
tickers_en = []
y_pred_en = []

y_trainval_true_en = []
y_trainval_pred_en = []

enet_param_grid = {
    'alpha':  np.logspace(np.log10(0.0008), np.log10(0.0005), num=12)
}

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):

    best_mse = float('inf')
    best_params = None

    for params in ParameterGrid(enet_param_grid):
        enet = ElasticNet(**params, max_iter=10000)
        enet.fit(x_train[covariates], y_train)
        y_val_pred = enet.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)

        if mse < best_mse:
            best_mse = mse
            best_params = params

    best_lambda = best_params['alpha']
    print(f"meilleur lambda {best_lambda}")
    best_lambdas.append(best_lambda)
    
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    en_final = ElasticNet(alpha=best_lambda, l1_ratio=0.5, max_iter=10000)
    en_final.fit(x_trainval, y_trainval)

    y_trainval_pred = en_final.predict(x_trainval)
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_in_sample_list_en.append(r2_in)

    y_test_pred = en_final.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)  
    r2_test_list_en.append(r2_out)


    print(f"R² in-sample : {r2_in:.6f} | R² test : {r2_out:.6f}")

    dates_en.append(x_test["Date"])
    tickers_en.append(x_test["Ticker"])
    y_pred_en.append(y_test_pred)
    y_trainval_true_en.append(y_trainval)
    y_trainval_pred_en.append(y_trainval_pred)

dates_en = np.concatenate(dates_en)
tickers_en = np.concatenate(tickers_en)
y_pred_en = np.concatenate(y_pred_en)
y_trainval_true_en = np.concatenate(y_trainval_true_en)
y_trainval_pred_en = np.concatenate(y_trainval_pred_en)

df_results_en = pd.DataFrame({
    "Date": dates_en,
    "Ticker": tickers_en,
    "y_pred_en": y_pred_en
})

r2_insample_en = r2(y_trainval_true_en, y_trainval_pred_en)
r2_oos_en = r2(y_true, y_pred_en)

r2_results = {
    "r2_in_sample_en": r2_insample_en,
    "r2_oos_global_en": r2_oos_en
}

  0%|          | 0/3 [00:00<?, ?it/s]

meilleur lambda 0.0008000000000000004


 33%|███▎      | 1/3 [00:34<01:09, 34.60s/it]

R² in-sample : 0.032678 | R² test : 0.037587
meilleur lambda 0.0008000000000000004


 67%|██████▋   | 2/3 [01:47<00:56, 56.87s/it]

R² in-sample : 0.032353 | R² test : 0.025731
meilleur lambda 0.0004999999999999999


100%|██████████| 3/3 [02:13<00:00, 44.41s/it]

R² in-sample : 0.035312 | R² test : 0.023347


In [ ]:
"""
RF : Random Forest
Hyperparamètres : Comme nous avons plus de paramètres nous codonds une grille de paramètres. 
- n_estimators : nombre d’arbres dans la forêt. 
- max_depth : profondeur maximale de chaque arbre. Limiter la profondeur permet de réduire l'overfitting. 
- min_samples_leaf : nombre minimal d’échantillons requis dans une feuille terminale. Augmenter cette valeur rend les arbres plus 
simples et limite l’overfitting.
- max_features : nombre de variables considérées pour choisir la meilleure séparation à chaque nœud.
"""

param_grid_rf = {
    'n_estimators': [100, 125, 150],
    'max_depth': [5, 6, 7],
    'min_samples_leaf': [1, 2, 3],
    'max_features': ['log2', None]
}

r2_in_sample_list_rf = []
r2_test_list_rf = []
best_params_rf = []
mse_val_grids_rf = []

dates_rf = []
tickers_rf = []
y_pred_rf = []

y_trainval_true_rf = []
y_trainval_pred_rf = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None 
    mse_grid = []

    for params in ParameterGrid(param_grid_rf):
        rf = RandomForestRegressor(
            **params,
            n_jobs=-1,
            random_state=0
        )
        rf.fit(x_train[covariates], y_train)
        y_val_pred = rf.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))

        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_rf.append(mse_grid)
    best_params_rf.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    rf_final = RandomForestRegressor(
        **best_params,
        n_jobs=-1,
        random_state=0
    )
    rf_final.fit(x_trainval, y_trainval)

    y_trainval_pred = rf_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_sample_list_rf.append(r2_in)

    y_test_pred = rf_final.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_test_list_rf.append(r2_out)

    tickers_rf.append(x_test["Ticker"])
    dates_rf.append(x_test["Date"])
    y_pred_rf.append(y_test_pred)
    y_trainval_true_rf.append(y_trainval)
    y_trainval_pred_rf.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² test : {r2_out:.6f}")

dates_rf = np.concatenate(dates_rf)
tickers_rf = np.concatenate(tickers_rf)
y_pred_rf = np.concatenate(y_pred_rf)
y_trainval_true_rf = np.concatenate(y_trainval_true_rf)
y_trainval_pred_rf = np.concatenate(y_trainval_pred_rf)

df_results_rf = pd.DataFrame({
    "Date": dates_rf,
    "Ticker" : tickers_rf,
    "y_pred_rf": y_pred_rf
})

r2_insample_rf = r2(y_trainval_true_rf, y_trainval_pred_rf)
r2_oos_rf = r2(y_true, y_pred_rf)


r2_results = {
    "r2_in_sample_rf": r2_insample_rf,
    "r2_oos_global_rf": r2_oos_rf
}

  0%|          | 0/3 [00:00<?, ?it/s]


Split 1 : meilleurs params = {'max_depth': 7, 'max_features': 'log2', 'min_samples_leaf': 3, 'n_estimators': 100} (MSE val = 0.002980)


 33%|███▎      | 1/3 [02:15<04:30, 135.16s/it]

R² in-sample : 0.119862 | R² test : 0.032320

Split 2 : meilleurs params = {'max_depth': 7, 'max_features': None, 'min_samples_leaf': 3, 'n_estimators': 100} (MSE val = 0.003386)


 67%|██████▋   | 2/3 [05:34<02:52, 172.77s/it]

R² in-sample : 0.158560 | R² test : 0.033139

Split 3 : meilleurs params = {'max_depth': 7, 'max_features': None, 'min_samples_leaf': 1, 'n_estimators': 100} (MSE val = 0.006771)


100%|██████████| 3/3 [08:36<00:00, 172.12s/it]

R² in-sample : 0.167599 | R² test : 0.028765


In [ ]:
"""
GBRT : Gradient Boosted Regression Trees
Hyperparamètres : 
- n_estimators : nombre d’arbres successifs dans le modèle. 
- learning_rate : taux d’apprentissage. Chaque nouvel arbre corrige les erreurs des précédents en suivant ce coefficient.
- max_depth : profondeur maximale de chaque arbre de base. 
- loss : fonction de perte utilisée pour ajuster les arbres 
- alpha : paramètre spécifique huber qui contrôle la sensibilité aux outliers.
"""

param_grid_gbrt = {
    'n_estimators': [200],     
    'learning_rate': [0.01], 
    'max_depth': [3],                 
    'loss': ['huber'],
    'alpha': [0.9]
}

r2_in_sample_list_gbrt = []
r2_test_list_gbrt = []
best_params_gbrt = []
mse_val_grids_gbrt = []

dates_gbrt = []
tickers_gbrt = []
y_pred_gbrt = []

y_trainval_true_gbrt = []
y_trainval_pred_gbrt = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    for params in ParameterGrid(param_grid_gbrt):
        gbrt = GradientBoostingRegressor(**params, random_state=0)
        gbrt.fit(x_train[covariates], y_train)
        y_val_pred = gbrt.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))

        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_gbrt.append(mse_grid)
    best_params_gbrt.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    gbrt_final = GradientBoostingRegressor(
        **best_params,
        random_state=0
    )
    gbrt_final.fit(x_trainval, y_trainval)

    y_trainval_pred = gbrt_final.predict(x_trainval)
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_in_sample_list_gbrt.append(r2_in)

    y_test_pred = gbrt_final.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_test_list_gbrt.append(r2_out)

    tickers_gbrt.append(x_test["Ticker"])
    dates_gbrt.append(x_test["Date"])
    y_pred_gbrt.append(y_test_pred)
    y_trainval_true_gbrt.append(y_trainval)
    y_trainval_pred_gbrt.append(y_trainval_pred)    

    print(f"R² in-sample : {r2_in:.6f} | R² test : {r2_out:.6f}")

dates_gbrt = np.concatenate(dates_gbrt)
tickers_gbrt = np.concatenate(tickers_gbrt)
y_pred_gbrt = np.concatenate(y_pred_gbrt)
y_trainval_true_gbrt = np.concatenate(y_trainval_true_gbrt)
y_trainval_pred_gbrt = np.concatenate(y_trainval_pred_gbrt)

df_results_gbrt = pd.DataFrame({
    "Date": dates_gbrt,
    "Ticker" : tickers_gbrt,
    "y_pred_gbrt": y_pred_gbrt
})

r2_insample_gbrt = r2(y_trainval_true_gbrt, y_trainval_pred_gbrt)
r2_oos_gbrt = r2(y_true, y_pred_gbrt)

r2_results = {
    "r2_in_sample_gbrt": r2_insample_gbrt,
    "r2_oos_global_gbrt": r2_oos_gbrt
}

  0%|          | 0/3 [00:00<?, ?it/s]


Split 1 : meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 3, 'n_estimators': 200} (MSE val = 0.003138)


 33%|███▎      | 1/3 [00:35<01:10, 35.18s/it]

R² in-sample : 0.041235 | R² test : 0.033550

Split 2 : meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 3, 'n_estimators': 200} (MSE val = 0.003482)


 67%|██████▋   | 2/3 [01:10<00:35, 35.47s/it]

R² in-sample : 0.040588 | R² test : 0.024566

Split 3 : meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 3, 'n_estimators': 200} (MSE val = 0.006852)


100%|██████████| 3/3 [01:48<00:00, 36.17s/it]

R² in-sample : 0.040664 | R² test : 0.017164


In [ ]:
#prédictions : all model 
df_predict = df_results_ols[["Date", "Ticker"]].copy()
df_predict = df_predict.merge(df_results_pls, on=["Date", "Ticker"], how="inner")
df_predict = df_predict.merge(df_results_pcr, on=["Date", "Ticker"], how="inner")
df_predict = df_predict.merge(df_results_en, on=["Date", "Ticker"], how="inner")
df_predict = df_predict.merge(df_results_rf, on=["Date", "Ticker"], how="inner")

In [26]:
#r2 
for key, value in r2_results.items():
    print


In [ ]:
df_predict.to_excel("df_predict.xlsx")

III. RESULTATS

In [ ]:
df_predict = df_results_ols.copy()
df_predict = df_predict.merge(df_results_pls, on=["Date", "Ticker"], how="inner")
df_predict = df_predict.merge(df_results_en, on=["Date", "Ticker"], how="inner")
df_predict = df_predict.merge(df_results_rf, on=["Date", "Ticker"], how="inner")
df_predict = df_predict.merge(df_results_gbrt, on=["Date", "Ticker"], how="inner")

In [ ]:
print(df_predict.head())

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
#PREMIER TABLEAU : R² OOS Monthly

#* 100 pcq on est en décimal 
R2_monthly_sum = {
    'OLS' : df_r2_monthly["R2_OLS"].mean() * 100,
    'PLS' : df_r2_monthly_pls["R2_PLS"].mean() * 100,
    'PCR' : df_r2_monthly_pcr["R2_PCR"].mean() * 100, 
    'Enet' : df_r2_monthly_en["R2_ENET"].mean() * 100,
    'RF' : df_r2_monthly_rf["R2_RF"].mean() * 100,
    'GBRT' : df_r2_monthly_gbrt["R2_GBRT"].mean() * 100
}


#tableau 
# Transformer le dictionnaire en DataFrame pour un tableau
df_table = pd.DataFrame.from_dict(R2_monthly_sum, orient='index', columns=['Mean_R2_OOS'])
print("=== Tableau résumé R² OOS Monthly ===")
print(df_table)



In [ ]:
import matplotlib.pyplot as plt

models = list(R2_monthly_sum.keys())
values = list(R2_monthly_sum.values())

plt.figure(figsize=(8,5))
bars = plt.bar(models, values, edgecolor='black', width=0.35)  # width < 1 pour des barres plus fines

# Titre et labels
plt.title('Comparaison des modèles (moyenne mensuelle)', fontsize=14, fontweight='bold')
plt.ylabel('Mean monthly $R^2_{OOS}$ (%)', fontsize=12)

# Valeurs au-dessus des barres
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.05,
             f'{height:.2f}', ha='center', va='bottom', fontsize=10)

# Couleur personnalisée
for bar in bars:
    bar.set_color('#4C72B0')

# Style épuré
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
#R2 benchmark HA 

def compute_oos_r_square(actual, y_benchmark, y_pred):
    MSFE_benchmark = mean_squared_error(y_benchmark, actual)
    MSFE_pred = mean_squared_error(y_pred, actual)
    return 1 - MSFE_pred / MSFE_benchmark

In [ ]:
#HA Benchmark 
actual_test = all_y_true_ols          # identique pour tous les modèles
y_pred_HA = all_y_pred_HA

#Prédictions concaténées dans un dictionnaire
model_predictions = {
    "OLS": all_y_pred_ols,
    "PLS": all_y_pred_pls,
    "PCR": all_y_pred_pcr,
    "ENet": all_y_pred_en,
    "RF": all_y_pred_rf,
    "GBRT": all_y_pred_gbrt
}

#Calcul des métriques
results = []

for model_name, y_pred in model_predictions.items():
    r2_oos = compute_oos_r_square(actual_test, y_pred_HA, y_pred) * 100  # en %
    results.append([model_name, r2_oos])

results.append(["HA", 0.0])

#df final 
df_r2_oos = pd.DataFrame(results, columns=["Model", "OOS_R2(%)"])

print(df_r2_oos)